In [1]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 1s (7,624 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package fonts-nanum.
(Reading database ... 121703 files and dire

# 데이터 불러오기

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rc('font', family='NanumBarunGothic')

!pip install tspoon

import tspoon as tsp

In [3]:
!pip install xmltodict
import pandas as pd

import requests
import xml.etree.ElementTree as ET
import xml.dom.minidom
import xmltodict

import json
from urllib.request import urlopen

def getECOS(topic,period,beg,end,item1='?',item2='?',item3='?',item4='?', df=None):
    key = ''
    url = 'https://ecos.bok.or.kr/api/StatisticSearch/'+'OIT9OYFOZDZWLHSBQJWU'+'/xml/kr/'+str(1)+'/'+str(50000)+'/'+topic+'/'+period+'/'+beg+'/'+end \
            +'/'+item1+'/'+item2+'/'+item3+'/'+item4
    # print(url)
    ## call OpenAPI URL
    response = requests.get(url)

    ## get API return value upon the http request
    if response.status_code == 200:
        try:
            contents = response.text
            ecosRoot = ET.fromstring(contents)

            # error check
            if ecosRoot[0].text[:4] in ("INFO","ERRO"):
                print(ecosRoot[0].text + " : " + ecosRoot[1].text)

            else:
                #print(contents)
                #dom = xml.dom.minidom.parse(xml_fname)
                dom = xml.dom.minidom.parseString(contents)
                pretty_xml_as_string = dom.toprettyxml(indent=" ")
                # print(pretty_xml_as_string)
                dic = xmltodict.parse(pretty_xml_as_string)

                n = int(dic['StatisticSearch']['list_total_count']['#text'])
                df_ecos = pd.DataFrame(index = [dic['StatisticSearch']['row'][i]['TIME'] for i in range(n)])
                df_ecos[dic['StatisticSearch']['row'][0]['ITEM_NAME1']] = [float(dic['StatisticSearch']['row'][i]['DATA_VALUE']) for i in range(n)]

                if type(df)==type(pd.DataFrame()):
                    df_ecos = df.merge(df_ecos, left_index=True, right_index=True, how='left')

                return df_ecos

        except Exception as e:
            print(str(e))


def getKOSIS(table,period,beg,end,item, orgId='101', obj1='',obj2='',obj3='',obj4='',obj5='',obj6='',obj7='',obj8='', title=None, title_no='0', debug=False):
     # Korean Statistics
    key = 'MzhiMmQwZTYyMWZiNGM1ZGFkZDJmZTI2OTE1OGU2NzQ='
    url = 'https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey='+key \
            +'&orgId='+orgId\
            +'&tblId='+table \
            +'&prdSe='+period \
            +'&startPrdDe='+beg \
            +'&endPrdDe='+end \
            +'&itmId='+item \
            +'&objL1='+obj1 \
            +'&objL2='+obj2 \
            +'&objL3='+obj3 \
            +'&objL4='+obj4 \
            +'&objL5='+obj5 \
            +'&objL6='+obj6 \
            +'&objL7='+obj7 \
            +'&objL8='+obj8 \
            +'&format=json&jsonVD=Y&outputFields=ITM_NM+PRD_DE+DT'
    res = requests.get(url)
    py_json = res.json()

    data = []
    for v in py_json:
        value = []
        value.append(v['PRD_DE'])

        # ✅ DT 값이 '-' 또는 ''일 때 NaN으로 처리
        try:
            value.append(float(v['DT']))
        except (ValueError, KeyError):
            value.append(np.nan)

        data.append(value)

    df = pd.DataFrame(data, columns=['PRD_DE', 'DT'])

    if debug:
        print(df.head())

    return df


    #get json data from url
    with urlopen(url) as url_:
        json_file = url_.read()

    py_json = json.loads(json_file.decode('utf-8'))


    #data transformation
    data = []

    for i, v in enumerate(py_json):
        value = []
        value.append(v['PRD_DE'])
        value.append(float(v['DT']))

        data.append(value)

    if title is not None:
        title = str(title)
    elif title_no=='0':
        title = v['ITM_NM_ENG']
    else:
        title = v['C'+title_no+'_NM_ENG']

    df_kosis = pd.DataFrame({v['PRD_SE']:[x[0] for x in data], title:[x[1] for x in data]}).set_index(v['PRD_SE'])

    if debug:
        return v

    return df_kosis

In [4]:
import pandas as pd
import numpy as np
import os
import seaborn as sns

In [5]:
from google.colab import drive
drive.mount('/content/gdrive/', force_remount=True)

Mounted at /content/gdrive/


In [6]:
# change directory to where you have your data file!!
%cd gdrive/MyDrive/GDP/데이터

/content/gdrive/MyDrive/GDP/데이터


## GDP

In [7]:
gdp = getECOS('200Y104', 'Q', '2015Q1', '2025Q2','110310').reset_index()
gdp

,index,전기장비 제조업
0,2015Q1,7301.1
1,2015Q2,7317.3
2,2015Q3,7135.0
3,2015Q4,6629.3
4,2016Q1,6644.5
5,2016Q2,6572.8
6,2016Q3,6739.4
7,2016Q4,6765.3
8,2017Q1,7320.1
9,2017Q2,7310.8


## PPI

In [8]:
ppi = getECOS('404Y014','Q','2015Q1','2025Q2','310AA').reset_index()
ppi

,index,전기장비
0,2015Q1,98.29
1,2015Q2,97.68
2,2015Q3,97.68
3,2015Q4,97.13
4,2016Q1,97.07
5,2016Q2,96.80
6,2016Q3,96.29
7,2016Q4,97.01
8,2017Q1,98.04
9,2017Q2,97.72


In [9]:
ppi = ppi.rename(columns={'전기장비':'전기장비ppi'})

##BSI

In [10]:
업황실적BSI = getECOS('512Y007','M','201501','202506','AA','C2800').reset_index()
매출실적BSI = getECOS('512Y007','M','201501','202506','AB','C2800').reset_index()
채산성실적BSI = getECOS('512Y007','M','201501','202506','AE','C2800').reset_index()
수출실적BSI= getECOS('512Y007','M','201501','202506','AM','C2800').reset_index()
내수판매실적BSI= getECOS('512Y007','M','201501','202506','AL','C2800').reset_index()
생산실적BSI= getECOS('512Y007','M','201501','202506','AC','C2800').reset_index()
신규수주실적BSI= getECOS('512Y007','M','201501','202506','AD','C2800').reset_index()
제품재고실적BSI= getECOS('512Y007','M','201501','202506','AG','C2800').reset_index()
가동률실적BSI= getECOS('512Y007','M','201501','202506','AK','C2800').reset_index()
생산설비실적BSI= getECOS('512Y007','M','201501','202506','AH','C2800').reset_index()
설비투자실적BSI= getECOS('512Y007','M','201501','202506','AI','C2800').reset_index()
자금사정실적BSI = getECOS('512Y007','M','201501','202506','AO','C2800').reset_index()
인력사정실적BSI = getECOS('512Y007','M','201501','202506','AJ','C2800').reset_index()

업황실적BSI = 업황실적BSI.rename(columns={'업황실적BSI 1)':'업황실적BSI'})
매출실적BSI = 매출실적BSI.rename(columns={'매출실적BSI 2)':'매출실적BSI'})
채산성실적BSI = 채산성실적BSI.rename(columns={'채산성실적BSI 6)':'채산성실적BSI'})
수출실적BSI = 수출실적BSI.rename(columns={'수출실적BSI 2)':'수출실적BSI'})
내수판매실적BSI = 내수판매실적BSI.rename(columns={'내수판매실적BSI 2)':'내수판매실적BSI'})
생산실적BSI = 생산실적BSI.rename(columns={'생산실적BSI 2)':'생산실적BSI'})
신규수주실적BSI = 신규수주실적BSI.rename(columns={'신규수주실적BSI 2)':'신규수주실적BSI'})
제품재고실적BSI = 제품재고실적BSI.rename(columns={'제품재고실적BSI 3)':'제품재고실적BSI'})
가동률실적BSI = 가동률실적BSI.rename(columns={'가동률실적BSI 4)':'가동률실적BSI'})
생산설비실적BSI = 생산설비실적BSI.rename(columns={'생산설비실적BSI 3)':'생산설비실적BSI'})
설비투자실적BSI = 설비투자실적BSI.rename(columns={'설비투자실적BSI 5)':'설비투자실적BSI '})
자금사정실적BSI = 자금사정실적BSI.rename(columns={'자금사정실적BSI 6)':'자금사정실적BSI'})
인력사정실적BSI = 인력사정실적BSI.rename(columns={'인력사정실적BSI 3)':'인력사정실적BSI'})


In [11]:
from functools import reduce

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [업황실적BSI, 매출실적BSI,수출실적BSI, 내수판매실적BSI, 생산실적BSI, 신규수주실적BSI, 제품재고실적BSI, 가동률실적BSI,설비투자실적BSI, 채산성실적BSI, 자금사정실적BSI, 인력사정실적BSI]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
df_merged = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
df_merged

,index,업황실적BSI,매출실적BSI,수출실적BSI,내수판매실적BSI,생산실적BSI,신규수주실적BSI,제품재고실적BSI,가동률실적BSI,설비투자실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI
0,201501,61.0,85.0,92.0,79.0,86.0,85.0,103.0,92.0,91.0,86.0,86.0,88.0
1,201502,76.0,95.0,96.0,91.0,92.0,93.0,105.0,86.0,97.0,86.0,85.0,97.0
2,201503,66.0,79.0,72.0,81.0,85.0,79.0,104.0,78.0,95.0,86.0,85.0,99.0
3,201504,78.0,88.0,81.0,89.0,86.0,88.0,104.0,89.0,90.0,93.0,85.0,97.0
4,201505,71.0,91.0,86.0,87.0,90.0,90.0,110.0,90.0,91.0,91.0,91.0,96.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,202502,78.0,84.0,105.0,76.0,92.0,87.0,103.0,92.0,90.0,92.0,98.0,87.0
122,202503,73.0,81.0,84.0,76.0,87.0,81.0,108.0,89.0,92.0,84.0,85.0,95.0
123,202504,64.0,69.0,72.0,69.0,80.0,66.0,107.0,85.0,92.0,80.0,82.0,97.0
124,202505,65.0,71.0,78.0,70.0,83.0,71.0,102.0,84.0,89.0,84.0,84.0,98.0


In [12]:
df_merged.isnull().sum()

,0
index,0
업황실적BSI,0
매출실적BSI,0
수출실적BSI,0
내수판매실적BSI,0
생산실적BSI,0
신규수주실적BSI,0
제품재고실적BSI,0
가동률실적BSI,0
설비투자실적BSI,0


In [13]:
import pandas as pd

# index가 숫자라면 문자열로 변환
df_merged['index'] = df_merged['index'].astype(str)

# 연도와 월 분리
df_merged['연도'] = df_merged['index'].str[:4].astype(int)
df_merged['월'] = df_merged['index'].str[4:6].astype(int)

# 월 → 분기 변환 함수
def month_to_quarter(month):
    if month in [1, 2, 3]:
        return 'Q1'
    elif month in [4, 5, 6]:
        return 'Q2'
    elif month in [7, 8, 9]:
        return 'Q3'
    else:
        return 'Q4'

# 분기 계산
df_merged['분기'] = df_merged['월'].apply(month_to_quarter)

# 분기 인덱스 생성 (예: 2015Q1)
df_merged['index'] = df_merged['연도'].astype(str) + df_merged['분기']

# 분기별 평균 집계
df_quarter = df_merged.groupby('index').mean(numeric_only=True).reset_index()

# 결과 확인
print(df_quarter.head())


    index    업황실적BSI    매출실적BSI    수출실적BSI  내수판매실적BSI    생산실적BSI  신규수주실적BSI  \
0  2015Q1  67.666667  86.333333  86.666667  83.666667  87.666667  85.666667   
1  2015Q2  70.333333  85.666667  81.666667  82.333333  84.666667  84.666667   
2  2015Q3  60.000000  72.666667  81.000000  68.333333  78.666667  75.000000   
3  2015Q4  60.000000  70.000000  76.000000  64.333333  80.000000  74.333333   
4  2016Q1  59.333333  69.666667  74.000000  66.000000  74.333333  69.000000   

    제품재고실적BSI   가동률실적BSI  설비투자실적BSI    채산성실적BSI  자금사정실적BSI  인력사정실적BSI      연도  \
0  104.000000  85.333333   94.333333  86.000000  85.333333  94.666667  2015.0   
1  106.666667  84.666667   89.000000  88.000000  85.333333  97.333333  2015.0   
2  109.000000  78.333333   91.666667  82.000000  83.666667  97.333333  2015.0   
3  107.000000  76.333333   89.333333  79.333333  82.333333  99.000000  2015.0   
4  107.666667  73.666667   91.333333  84.666667  82.000000  95.333333  2016.0   

      월  
0   2.0  
1   5.0  
2   8.0 

In [14]:
df_quarter.head()

,index,업황실적BSI,매출실적BSI,수출실적BSI,내수판매실적BSI,생산실적BSI,신규수주실적BSI,제품재고실적BSI,가동률실적BSI,설비투자실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,연도,월
0,2015Q1,67.666667,86.333333,86.666667,83.666667,87.666667,85.666667,104.000000,85.333333,94.333333,86.000000,85.333333,94.666667,2015.0,2.0
1,2015Q2,70.333333,85.666667,81.666667,82.333333,84.666667,84.666667,106.666667,84.666667,89.000000,88.000000,85.333333,97.333333,2015.0,5.0
2,2015Q3,60.000000,72.666667,81.000000,68.333333,78.666667,75.000000,109.000000,78.333333,91.666667,82.000000,83.666667,97.333333,2015.0,8.0
3,2015Q4,60.000000,70.000000,76.000000,64.333333,80.000000,74.333333,107.000000,76.333333,89.333333,79.333333,82.333333,99.000000,2015.0,11.0
4,2016Q1,59.333333,69.666667,74.000000,66.000000,74.333333,69.000000,107.666667,73.666667,91.333333,84.666667,82.000000,95.333333,2016.0,2.0


In [15]:
df_quarter = df_quarter.drop(columns=['연도', '월'], errors='ignore')


In [16]:
dfs = [ppi, gdp]
df_final = df_quarter.copy()
key_col = 'index'
for df in dfs:
    df_final = df_final.merge(df, on=key_col, how='left')

df_final

,index,업황실적BSI,매출실적BSI,수출실적BSI,내수판매실적BSI,생산실적BSI,신규수주실적BSI,제품재고실적BSI,가동률실적BSI,설비투자실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,전기장비ppi,전기장비 제조업
0,2015Q1,67.666667,86.333333,86.666667,83.666667,87.666667,85.666667,104.000000,85.333333,94.333333,86.000000,85.333333,94.666667,98.29,7301.1
1,2015Q2,70.333333,85.666667,81.666667,82.333333,84.666667,84.666667,106.666667,84.666667,89.000000,88.000000,85.333333,97.333333,97.68,7317.3
2,2015Q3,60.000000,72.666667,81.000000,68.333333,78.666667,75.000000,109.000000,78.333333,91.666667,82.000000,83.666667,97.333333,97.68,7135.0
3,2015Q4,60.000000,70.000000,76.000000,64.333333,80.000000,74.333333,107.000000,76.333333,89.333333,79.333333,82.333333,99.000000,97.13,6629.3
4,2016Q1,59.333333,69.666667,74.000000,66.000000,74.333333,69.000000,107.666667,73.666667,91.333333,84.666667,82.000000,95.333333,97.07,6644.5
5,2016Q2,62.666667,72.333333,81.000000,70.333333,77.000000,72.000000,110.000000,78.333333,86.666667,84.666667,86.333333,102.000000,96.80,6572.8
6,2016Q3,62.666667,69.333333,77.666667,69.666667,74.666667,68.666667,107.666667,74.333333,90.000000,84.333333,88.333333,100.666667,96.29,6739.4
7,2016Q4,68.333333,74.333333,85.000000,72.000000,81.000000,76.000000,103.666667,83.666667,91.666667,85.333333,88.000000,97.333333,97.01,6765.3
8,2017Q1,69.333333,79.333333,88.333333,78.000000,87.000000,81.333333,107.666667,85.000000,93.666667,77.333333,81.000000,96.666667,98.04,7320.1
9,2017Q2,80.333333,84.666667,91.000000,81.000000,93.333333,89.000000,108.333333,87.666667,95.333333,88.000000,87.666667,103.333333,97.72,7310.8


In [17]:
df_final.to_csv("전기장비1차.csv")

## 생산지수

In [18]:
생산지수 = getKOSIS('DT_1F02001','Q','201501','202502','T20','101','00','C28')

def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

생산지수['index']=생산지수['PRD_DE'].apply(to_quarter)
생산지수 = 생산지수.drop(columns=["PRD_DE"])
생산지수 = 생산지수.rename(columns={'DT': '생산지수'})
생산지수

,생산지수,index
0,90.577,2015Q1
1,90.360,2015Q2
2,93.073,2015Q3
3,94.494,2015Q4
4,95.813,2016Q1
5,92.740,2016Q2
6,94.348,2016Q3
7,97.281,2016Q4
8,96.966,2017Q1
9,98.666,2017Q2


## 내수/수출출하지수

https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=MGNkMzY3MTdhNjZlMTNiZDMxYjU4NDA4NTdjZWU0YmQ=&itmId=T21+&objL1=C28+&objL2=&objL3=&objL4=&objL5=&objL6=&objL7=&objL8=&format=json&jsonVD=Y&prdSe=Q&startPrdDe=201501&endPrdDe=202502&outputFields=OBJ_NM+NM+ITM_NM+PRD_DE+&orgId=101&tblId=DT_1F02016

In [19]:
수출출하지수 = getKOSIS('DT_1F02016','Q','201501','202502','T21','101','C28')

def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

수출출하지수['index']=수출출하지수['PRD_DE'].apply(to_quarter)
수출출하지수 = 수출출하지수.drop(columns=["PRD_DE"])
수출출하지수 = 수출출하지수.rename(columns={'DT': '수출출하지수'})
수출출하지수

,수출출하지수,index
0,90.071,2015Q1
1,91.428,2015Q2
2,93.093,2015Q3
3,94.480,2015Q4
4,99.969,2016Q1
5,95.899,2016Q2
6,97.564,2016Q3
7,97.718,2016Q4
8,95.961,2017Q1
9,96.731,2017Q2


In [20]:
내수출하지수 = getKOSIS('DT_1F02016','Q','201501','202502','T20','101','C28')
def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

내수출하지수['index']=내수출하지수['PRD_DE'].apply(to_quarter)
내수출하지수 = 내수출하지수.drop(columns=["PRD_DE"])
내수출하지수 = 내수출하지수.rename(columns={'DT': '내수출하지수'})
내수출하지수

,내수출하지수,index
0,90.381,2015Q1
1,90.138,2015Q2
2,92.510,2015Q3
3,91.658,2015Q4
4,90.381,2016Q1
5,89.712,2016Q2
6,90.442,2016Q3
7,94.152,2016Q4
8,94.669,2017Q1
9,94.244,2017Q2


https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=MGNkMzY3MTdhNjZlMTNiZDMxYjU4NDA4NTdjZWU0YmQ=&itmId=T10+T30+&objL1=C28+&objL2=&objL3=&objL4=&objL5=&objL6=&objL7=&objL8=&format=json&jsonVD=Y&prdSe=Q&startPrdDe=201501&endPrdDe=202502&outputFields=OBJ_NM+NM+ITM_NM+PRD_DE+&orgId=101&tblId=DT_1F32001

## 생산능력지수 & 가동률지수

In [21]:
생산능력지수 = getKOSIS('DT_1F32001','Q','201501','202502','T10','101','C28')
def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

생산능력지수['index']=생산능력지수['PRD_DE'].apply(to_quarter)
생산능력지수 = 생산능력지수.drop(columns=["PRD_DE"])
생산능력지수 = 생산능력지수.rename(columns={'DT': '생산능력지수'})
생산능력지수

,생산능력지수,index
0,93.177,2015Q1
1,93.333,2015Q2
2,94.178,2015Q3
3,94.929,2015Q4
4,94.210,2016Q1
5,93.865,2016Q2
6,94.773,2016Q3
7,94.053,2016Q4
8,95.211,2017Q1
9,96.369,2017Q2


In [22]:
가동률지수 = getKOSIS('DT_1F32001','Q','201501','202502','T30','101','C28')
def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

가동률지수['index']=가동률지수['PRD_DE'].apply(to_quarter)
가동률지수 = 가동률지수.drop(columns=["PRD_DE"])
가동률지수 = 가동률지수.rename(columns={'DT': '가동률지수'})
가동률지수

,가동률지수,index
0,97.858,2015Q1
1,97.522,2015Q2
2,100.217,2015Q3
3,99.958,2015Q4
4,101.547,2016Q1
5,99.093,2016Q2
6,100.701,2016Q3
7,103.622,2016Q4
8,102.813,2017Q1
9,102.724,2017Q2


## 기업데이터 (인건비 + 에빗다)

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [24]:
기업 = pd.read_excel("Data_통합.xlsx")

In [25]:
import pandas as pd

file_path = "Data_통합.xlsx"

# 엑셀 파일 객체 생성
xls = pd.ExcelFile(file_path)

# 빈 딕셔너리 (시트별 데이터 저장용)
dfs = {}

# 모든 시트를 순회
for sheet in xls.sheet_names:
    print(f"Loading {sheet} ...")

    if sheet == "Sheet":
        # 첫 번째 시트는 첫 번째 줄이 컬럼명
        dfs[sheet] = pd.read_excel(file_path, sheet_name=sheet)
    else:
        # 다른 시트는 두 번째 줄부터 컬럼명
        dfs[sheet] = pd.read_excel(file_path, sheet_name=sheet, header=1)

    print(f"{sheet} shape: {dfs[sheet].shape}")


Loading Sheet ...
Sheet shape: (374340, 14)
Loading EBITDA ...
EBITDA shape: (7342, 112)
Loading 인건비 ...
인건비 shape: (7342, 112)
Loading 유무형리스자산의증가 ...
유무형리스자산의증가 shape: (7342, 112)
Loading 매출원가 ...
매출원가 shape: (7342, 112)
Loading 수익 ...
수익 shape: (7342, 112)


In [26]:
dfs["Sheet"]

,업체코드,종목코드,종목명,691005.업체명,691020.사업자번호,691050.상장일,691060.결산월,691450.KSIC-세세분류(11차),DATE,EBITDA,인건비,"유,무형,리스자산의증가",매출원가,수익(매출액)
0,N350605,A000020,동화약품,동화약품(주),110-81-00102,1976-03-24,12.0,C21212.합성의약품 및 기타 완제 의약품 제조업,2000-01-01,0.000000e+00,3829199.0,0.000000e+00,1.698668e+10,2.874157e+10
1,N350605,A000020,동화약품,동화약품(주),110-81-00102,1976-03-24,12.0,C21212.합성의약품 및 기타 완제 의약품 제조업,2000-04-01,0.000000e+00,4129827.0,0.000000e+00,2.042499e+10,3.281942e+10
2,N350605,A000020,동화약품,동화약품(주),110-81-00102,1976-03-24,12.0,C21212.합성의약품 및 기타 완제 의약품 제조업,2000-07-01,0.000000e+00,3811223.0,0.000000e+00,2.191622e+10,3.407650e+10
3,N350605,A000020,동화약품,동화약품(주),110-81-00102,1976-03-24,12.0,C21212.합성의약품 및 기타 완제 의약품 제조업,2000-10-01,4.886518e+09,7843796.0,4.438878e+09,2.584407e+10,4.259254e+10
4,N350605,A000020,동화약품,동화약품(주),110-81-00102,1976-03-24,12.0,C21212.합성의약품 및 기타 완제 의약품 제조업,2001-01-01,0.000000e+00,4329948.0,0.000000e+00,1.674465e+10,3.004641e+10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
374335,NMC8807,A950220,네오이뮨텍,(주)네오이뮨텍,NaN,2021-03-16,12.0,M70130.자연과학 및 공학 융합 연구개발업,2024-04-01,-9.589924e+09,-2940319.0,9.632000e+06,0.000000e+00,0.000000e+00
374336,NMC8807,A950220,네오이뮨텍,(주)네오이뮨텍,NaN,2021-03-16,12.0,M70130.자연과학 및 공학 융합 연구개발업,2024-07-01,-1.188333e+10,7760477.0,5.462700e+07,0.000000e+00,1.090765e+09
374337,NMC8807,A950220,네오이뮨텍,(주)네오이뮨텍,NaN,2021-03-16,12.0,M70130.자연과학 및 공학 융합 연구개발업,2024-10-01,3.202594e+10,-7760477.0,-1.113170e+08,0.000000e+00,-1.090765e+09
374338,NMC8807,A950220,네오이뮨텍,(주)네오이뮨텍,NaN,2021-03-16,12.0,M70130.자연과학 및 공학 융합 연구개발업,2025-01-01,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00


In [27]:
arr = dfs["Sheet"]['691450.KSIC-세세분류(11차)'].unique()
arr2 = pd.DataFrame(arr, columns=['산업분류'])
arr2

,산업분류
0,C21212.합성의약품 및 기타 완제 의약품 제조업
1,C31921.모터사이클 제조업
2,G47119.기타 대형 종합 소매업
3,K64992.지주회사
4,NaN
...,...
473,C29222.디지털 적층 성형기계 제조업
474,C15219.기타 신발 제조업
475,C29163.컨베이어장치 제조업
476,G45212.자동차용 전용 신품 부품 판매업


In [28]:
arr2.to_csv('기업데이터 산업.csv', encoding='utf-8-sig')

In [29]:
df1 = pd.read_csv('산업코드.csv')

In [30]:
df1

,산업 구분,산업코드
0,농림어업,A
1,광업,B
2,음식료품 제조업_식료품,C10
3,음식료품 제조업_음료,C11
4,음식료품 제조업_담배,C12
5,섬유 및 가죽제품 제조업_섬유,C13
6,섬유 및 가죽제품 제조업_의복모피,C14
7,섬유 및 가죽제품 제조업_가죽가방신발,C15
8,"목재, 종이, 인쇄 및 복제업_목재나무",C16
9,"목재, 종이, 인쇄 및 복제업_펄프종이",C17


In [31]:
def match_industry(row, code_dict):
    if pd.isna(row):
        return np.nan
    for code, industry in code_dict.items():
        if str(row).startswith(code):
            return industry
    return ''

code_dict = dict(zip(df1['산업코드'], df1['산업 구분']))

In [32]:
# df2['산업분류']에서 코드 부분만 추출해서 매핑
arr2['매핑한 산업'] = arr2['산업분류'].apply(lambda x: match_industry(x, code_dict))
arr2

,산업분류,매핑한 산업
0,C21212.합성의약품 및 기타 완제 의약품 제조업,화학물질 및 화학제품 제조업_의료물질의약품
1,C31921.모터사이클 제조업,운송장비 제조업_조선기타운수
2,G47119.기타 대형 종합 소매업,도소매업
3,K64992.지주회사,금융 및 보험업
4,NaN,NaN
...,...,...
473,C29222.디지털 적층 성형기계 제조업,기계 및 장비 제조업
474,C15219.기타 신발 제조업,섬유 및 가죽제품 제조업_가죽가방신발
475,C29163.컨베이어장치 제조업,기계 및 장비 제조업
476,G45212.자동차용 전용 신품 부품 판매업,도소매업


In [33]:
dfs["Sheet"]['매핑한 산업'] = dfs["Sheet"]['691450.KSIC-세세분류(11차)'].apply(lambda x: match_industry(x, code_dict))
dfs["Sheet"]

,업체코드,종목코드,종목명,691005.업체명,691020.사업자번호,691050.상장일,691060.결산월,691450.KSIC-세세분류(11차),DATE,EBITDA,인건비,"유,무형,리스자산의증가",매출원가,수익(매출액),매핑한 산업
0,N350605,A000020,동화약품,동화약품(주),110-81-00102,1976-03-24,12.0,C21212.합성의약품 및 기타 완제 의약품 제조업,2000-01-01,0.000000e+00,3829199.0,0.000000e+00,1.698668e+10,2.874157e+10,화학물질 및 화학제품 제조업_의료물질의약품
1,N350605,A000020,동화약품,동화약품(주),110-81-00102,1976-03-24,12.0,C21212.합성의약품 및 기타 완제 의약품 제조업,2000-04-01,0.000000e+00,4129827.0,0.000000e+00,2.042499e+10,3.281942e+10,화학물질 및 화학제품 제조업_의료물질의약품
2,N350605,A000020,동화약품,동화약품(주),110-81-00102,1976-03-24,12.0,C21212.합성의약품 및 기타 완제 의약품 제조업,2000-07-01,0.000000e+00,3811223.0,0.000000e+00,2.191622e+10,3.407650e+10,화학물질 및 화학제품 제조업_의료물질의약품
3,N350605,A000020,동화약품,동화약품(주),110-81-00102,1976-03-24,12.0,C21212.합성의약품 및 기타 완제 의약품 제조업,2000-10-01,4.886518e+09,7843796.0,4.438878e+09,2.584407e+10,4.259254e+10,화학물질 및 화학제품 제조업_의료물질의약품
4,N350605,A000020,동화약품,동화약품(주),110-81-00102,1976-03-24,12.0,C21212.합성의약품 및 기타 완제 의약품 제조업,2001-01-01,0.000000e+00,4329948.0,0.000000e+00,1.674465e+10,3.004641e+10,화학물질 및 화학제품 제조업_의료물질의약품
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
374335,NMC8807,A950220,네오이뮨텍,(주)네오이뮨텍,NaN,2021-03-16,12.0,M70130.자연과학 및 공학 융합 연구개발업,2024-04-01,-9.589924e+09,-2940319.0,9.632000e+06,0.000000e+00,0.000000e+00,"전문, 과학 및 기술관련 서비스업"
374336,NMC8807,A950220,네오이뮨텍,(주)네오이뮨텍,NaN,2021-03-16,12.0,M70130.자연과학 및 공학 융합 연구개발업,2024-07-01,-1.188333e+10,7760477.0,5.462700e+07,0.000000e+00,1.090765e+09,"전문, 과학 및 기술관련 서비스업"
374337,NMC8807,A950220,네오이뮨텍,(주)네오이뮨텍,NaN,2021-03-16,12.0,M70130.자연과학 및 공학 융합 연구개발업,2024-10-01,3.202594e+10,-7760477.0,-1.113170e+08,0.000000e+00,-1.090765e+09,"전문, 과학 및 기술관련 서비스업"
374338,NMC8807,A950220,네오이뮨텍,(주)네오이뮨텍,NaN,2021-03-16,12.0,M70130.자연과학 및 공학 융합 연구개발업,2025-01-01,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,"전문, 과학 및 기술관련 서비스업"


In [34]:
industry_list = [
   "전기장비 제조업","기계 및 장비 제조업","운송장비 제조업_자동차", "운송장비 제조업_조선기타운수", "기타 제조업 및 산업용 장비 수리업_가구","기타 제조업 및 산업용 장비 수리업_기타제품","전기업 + 가스, 증기 및 공기조절 공급업",
   "수도, 하수 및 폐기물 처리, 원료 재생업","건물건설 및 건축보수업","토목건설업","도소매업","숙박 및 음식점업","운수업"
]

filtered_df = dfs["Sheet"][dfs["Sheet"]['매핑한 산업'].isin(industry_list)]
filtered_df

,업체코드,종목코드,종목명,691005.업체명,691020.사업자번호,691050.상장일,691060.결산월,691450.KSIC-세세분류(11차),DATE,EBITDA,인건비,"유,무형,리스자산의증가",매출원가,수익(매출액),매핑한 산업
102,N320498,A000040,KR모터스,KR모터스(주),305-81-00020,1976-05-25,12.0,C31921.모터사이클 제조업,2000-01-01,0.000000e+00,544152.0,0.000000e+00,2.799049e+10,3.131630e+10,운송장비 제조업_조선기타운수
103,N320498,A000040,KR모터스,KR모터스(주),305-81-00020,1976-05-25,12.0,C31921.모터사이클 제조업,2000-04-01,0.000000e+00,529446.0,0.000000e+00,3.218369e+10,3.792470e+10,운송장비 제조업_조선기타운수
104,N320498,A000040,KR모터스,KR모터스(주),305-81-00020,1976-05-25,12.0,C31921.모터사이클 제조업,2000-07-01,0.000000e+00,530405.0,0.000000e+00,3.185522e+10,3.875848e+10,운송장비 제조업_조선기타운수
105,N320498,A000040,KR모터스,KR모터스(주),305-81-00020,1976-05-25,12.0,C31921.모터사이클 제조업,2000-10-01,-7.798581e+09,10143695.0,3.741770e+09,2.611673e+10,2.787019e+10,운송장비 제조업_조선기타운수
106,N320498,A000040,KR모터스,KR모터스(주),305-81-00020,1976-05-25,12.0,C31921.모터사이클 제조업,2001-01-01,0.000000e+00,537856.0,0.000000e+00,2.133264e+10,2.561618e+10,운송장비 제조업_조선기타운수
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373927,NJB0688,A950170,JTC,(주)제이티씨,NaN,2018-04-06,2.0,G47130.면세점,2024-04-01,1.109332e+10,5284616.0,3.226334e+09,1.960907e+10,7.749555e+10,도소매업
373928,NJB0688,A950170,JTC,(주)제이티씨,NaN,2018-04-06,2.0,G47130.면세점,2024-07-01,1.145466e+10,5532262.0,1.890192e+09,1.798675e+10,7.447296e+10,도소매업
373929,NJB0688,A950170,JTC,(주)제이티씨,NaN,2018-04-06,2.0,G47130.면세점,2024-10-01,-6.333496e+09,-3883023.0,-4.152853e+09,-2.321521e+10,-8.668787e+10,도소매업
373930,NJB0688,A950170,JTC,(주)제이티씨,NaN,2018-04-06,2.0,G47130.면세점,2025-01-01,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,도소매업


In [35]:
filtered_df = filtered_df.rename(columns={'691005.업체명':'업체명', '691020.사업자번호':'사업자번호',
                                    '691050.상장일':'상장번호', '691050.상장일':'상장일',
                                    '691060.결산월':'결산월', '691450.KSIC-세세분류(11차)':'산업세세분류',
                                    'DATE':'조사일'})
filtered_df.head()

,업체코드,종목코드,종목명,업체명,사업자번호,상장일,결산월,산업세세분류,조사일,EBITDA,인건비,"유,무형,리스자산의증가",매출원가,수익(매출액),매핑한 산업
102,N320498,A000040,KR모터스,KR모터스(주),305-81-00020,1976-05-25,12.0,C31921.모터사이클 제조업,2000-01-01,0.000000e+00,544152.0,0.000000e+00,2.799049e+10,3.131630e+10,운송장비 제조업_조선기타운수
103,N320498,A000040,KR모터스,KR모터스(주),305-81-00020,1976-05-25,12.0,C31921.모터사이클 제조업,2000-04-01,0.000000e+00,529446.0,0.000000e+00,3.218369e+10,3.792470e+10,운송장비 제조업_조선기타운수
104,N320498,A000040,KR모터스,KR모터스(주),305-81-00020,1976-05-25,12.0,C31921.모터사이클 제조업,2000-07-01,0.000000e+00,530405.0,0.000000e+00,3.185522e+10,3.875848e+10,운송장비 제조업_조선기타운수
105,N320498,A000040,KR모터스,KR모터스(주),305-81-00020,1976-05-25,12.0,C31921.모터사이클 제조업,2000-10-01,-7.798581e+09,10143695.0,3.741770e+09,2.611673e+10,2.787019e+10,운송장비 제조업_조선기타운수
106,N320498,A000040,KR모터스,KR모터스(주),305-81-00020,1976-05-25,12.0,C31921.모터사이클 제조업,2001-01-01,0.000000e+00,537856.0,0.000000e+00,2.133264e+10,2.561618e+10,운송장비 제조업_조선기타운수


In [36]:
filtered_df.to_csv('임지오기업.csv', encoding='utf-8-sig')

In [37]:
df = pd.read_csv('임지오기업.csv')

In [38]:
전기장비기업 = df[df['매핑한 산업'] == "전기장비 제조업"]

In [39]:
columns = ['EBITDA', '인건비']
grouped = 전기장비기업.groupby('조사일')[columns].sum()
grouped

,EBITDA,인건비
조사일,,
2000-01-01,0.000000e+00,11873990.0
2000-04-01,0.000000e+00,63903976.0
2000-07-01,0.000000e+00,-7221776.0
2000-10-01,1.422085e+12,743075745.0
2001-01-01,0.000000e+00,15667442.0
...,...,...
2024-04-01,5.880269e+11,550383950.0
2024-07-01,7.237220e+10,492519748.0
2024-10-01,1.550230e+11,585506854.0


In [40]:
grouped = grouped.reset_index()

In [41]:
grouped

,조사일,EBITDA,인건비
0,2000-01-01,0.000000e+00,11873990.0
1,2000-04-01,0.000000e+00,63903976.0
2,2000-07-01,0.000000e+00,-7221776.0
3,2000-10-01,1.422085e+12,743075745.0
4,2001-01-01,0.000000e+00,15667442.0
...,...,...,...
97,2024-04-01,5.880269e+11,550383950.0
98,2024-07-01,7.237220e+10,492519748.0
99,2024-10-01,1.550230e+11,585506854.0
100,2025-01-01,1.248370e+11,235882356.0


In [42]:
grouped['조사일']=pd.to_datetime(grouped['조사일'])

# 분기(Quarter) 단위로 변환
grouped["index"] = grouped["조사일"].dt.to_period("Q")
grouped = grouped.drop(columns=["조사일"])
print(grouped)

# # 조사일이 Period 타입(예: 2010Q1)이라고 가정
# mask = (grouped["index"] >= pd.Period("2010Q1")) & (grouped["index"] <= pd.Period("2025Q1"))
# 기업= grouped.loc[mask]

           EBITDA          인건비   index
0    0.000000e+00   11873990.0  2000Q1
1    0.000000e+00   63903976.0  2000Q2
2    0.000000e+00   -7221776.0  2000Q3
3    1.422085e+12  743075745.0  2000Q4
4    0.000000e+00   15667442.0  2001Q1
..            ...          ...     ...
97   5.880269e+11  550383950.0  2024Q2
98   7.237220e+10  492519748.0  2024Q3
99   1.550230e+11  585506854.0  2024Q4
100  1.248370e+11  235882356.0  2025Q1
101 -2.341485e+10 -235882356.0  2025Q2

[102 rows x 3 columns]


In [43]:
mask = (grouped["index"] >= pd.Period("2015Q1")) & (grouped["index"] <= pd.Period("2025Q2"))
기업= grouped.loc[mask]
기업
기업['합산'] = 기업['EBITDA']+기업['인건비']
기업

/tmp/ipython-input-2516843845.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  기업['합산'] = 기업['EBITDA']+기업['인건비']


,EBITDA,인건비,index,합산
60,2.596062e+11,1.686700e+08,2015Q1,2.597749e+11
61,2.846790e+11,1.546657e+08,2015Q2,2.848337e+11
62,3.663875e+11,1.607252e+08,2015Q3,3.665482e+11
63,1.716946e+11,4.209451e+08,2015Q4,1.721156e+11
64,-4.316049e+11,6.914076e+08,2016Q1,-4.309135e+11
65,2.069940e+11,1.461942e+08,2016Q2,2.071402e+11
66,1.598279e+11,1.370653e+08,2016Q3,1.599650e+11
67,4.143464e+11,7.003417e+08,2016Q4,4.150468e+11
68,2.254426e+11,1.457721e+08,2017Q1,2.255884e+11
69,3.773892e+11,1.413807e+08,2017Q2,3.775305e+11


In [44]:
기업 = 기업[['index'] + [col for col in 기업.columns if col != 'index']]
기업 = 기업.reset_index()

In [45]:
기업
기업 = 기업.drop(columns=['level_0'])


In [46]:
기업

,index,EBITDA,인건비,합산
0,2015Q1,2.596062e+11,1.686700e+08,2.597749e+11
1,2015Q2,2.846790e+11,1.546657e+08,2.848337e+11
2,2015Q3,3.663875e+11,1.607252e+08,3.665482e+11
3,2015Q4,1.716946e+11,4.209451e+08,1.721156e+11
4,2016Q1,-4.316049e+11,6.914076e+08,-4.309135e+11
5,2016Q2,2.069940e+11,1.461942e+08,2.071402e+11
6,2016Q3,1.598279e+11,1.370653e+08,1.599650e+11
7,2016Q4,4.143464e+11,7.003417e+08,4.150468e+11
8,2017Q1,2.254426e+11,1.457721e+08,2.255884e+11
9,2017Q2,3.773892e+11,1.413807e+08,3.775305e+11


In [47]:
가동률지수
생산능력지수
생산지수

생산지수['index']=생산지수['PRD_DE'].apply(to_quarter)
생산지수 = 생산지수.drop(columns=["PRD_DE"])
생산지수 = 생산지수.rename(columns={'DT': '생산지수'})
생산지수


KeyError: 'PRD_DE'

In [48]:
내수출하지수

,내수출하지수,index
0,90.381,2015Q1
1,90.138,2015Q2
2,92.510,2015Q3
3,91.658,2015Q4
4,90.381,2016Q1
5,89.712,2016Q2
6,90.442,2016Q3
7,94.152,2016Q4
8,94.669,2017Q1
9,94.244,2017Q2


In [ ]:
수출출하지수
수출출하지수['index']=수출출하지수['PRD_DE'].apply(to_quarter)
수출출하지수 = 수출출하지수.drop(columns=["PRD_DE"])
수출출하지수 = 수출출하지수.rename(columns={'DT': '수출출하지수'})
수출출하지수

## 전체 데이터 합산

In [49]:
from functools import reduce
import pandas as pd

# 모든 데이터프레임의 index 컬럼을 문자열로 변환
for df in [생산지수, 내수출하지수, 수출출하지수, 생산능력지수, 가동률지수, 기업]:
    df['index'] = df['index'].astype(str)

# 리스트에 넣고 합치기
dfs = [생산지수, 내수출하지수, 수출출하지수, 생산능력지수, 가동률지수, 기업]
df_merged = reduce(lambda left, right: pd.merge(left, right, on='index', how='outer'), dfs)


In [50]:
df_merged
df_merged = df_merged[['index'] + [col for col in df_merged.columns if col != 'index']]

In [51]:
df_merged

,index,생산지수,내수출하지수,수출출하지수,생산능력지수,가동률지수,EBITDA,인건비,합산
0,2015Q1,90.577,90.381,90.071,93.177,97.858,2.596062e+11,1.686700e+08,2.597749e+11
1,2015Q2,90.360,90.138,91.428,93.333,97.522,2.846790e+11,1.546657e+08,2.848337e+11
2,2015Q3,93.073,92.510,93.093,94.178,100.217,3.663875e+11,1.607252e+08,3.665482e+11
3,2015Q4,94.494,91.658,94.480,94.929,99.958,1.716946e+11,4.209451e+08,1.721156e+11
4,2016Q1,95.813,90.381,99.969,94.210,101.547,-4.316049e+11,6.914076e+08,-4.309135e+11
5,2016Q2,92.740,89.712,95.899,93.865,99.093,2.069940e+11,1.461942e+08,2.071402e+11
6,2016Q3,94.348,90.442,97.564,94.773,100.701,1.598279e+11,1.370653e+08,1.599650e+11
7,2016Q4,97.281,94.152,97.718,94.053,103.622,4.143464e+11,7.003417e+08,4.150468e+11
8,2017Q1,96.966,94.669,95.961,95.211,102.813,2.254426e+11,1.457721e+08,2.255884e+11
9,2017Q2,98.666,94.244,96.731,96.369,102.724,3.773892e+11,1.413807e+08,3.775305e+11


In [52]:
d = pd.read_csv("전기장비1차.csv")

In [53]:
from functools import reduce
import pandas as pd



# 리스트에 넣고 합치기
dfs = [df_merged,d]
merged = reduce(lambda left, right: pd.merge(left, right, on='index', how='outer'), dfs)

In [54]:
merged
merged = merged.drop(columns=["Unnamed: 0"])

In [55]:
merged.to_csv('전기장비.csv')

#데이터QoQ 변환

In [56]:
import pandas as pd

# df는 위의 데이터프레임
df_qoq = merged.copy()

# 분기 순서대로 정렬 (혹시 index가 문자열일 경우를 대비)
df_qoq = df_qoq.sort_values('index')

# 수치형 컬럼만 선택 (index 제외)
num_cols = df_qoq.select_dtypes(include=['number']).columns

# 전분기 대비 증감률 (%) 계산
df_qoq[num_cols] = df_qoq[num_cols].pct_change() * 100

# 결과 확인
print(df_qoq.head())


    index      생산지수    내수출하지수    수출출하지수    생산능력지수     가동률지수      EBITDA  \
0  2015Q1       NaN       NaN       NaN       NaN       NaN         NaN   
1  2015Q2 -0.239575 -0.268862  1.506589  0.167423 -0.343355    9.658010   
2  2015Q3  3.002435  2.631521  1.821105  0.905360  2.763479   28.701981   
3  2015Q4  1.526759 -0.920982  1.489908  0.797426 -0.258439  -53.138517   
4  2016Q1  1.395856 -1.393223  5.809695 -0.757408  1.589668 -351.379418   

          인건비          합산    업황실적BSI  ...   생산실적BSI  신규수주실적BSI  제품재고실적BSI  \
0         NaN         NaN        NaN  ...       NaN        NaN        NaN   
1   -8.302819    9.646348   3.940887  ... -3.422053  -1.167315   2.564103   
2    3.917862   28.688523 -14.691943  ... -7.086614 -11.417323   2.187500   
3  161.903548  -53.044225   0.000000  ...  1.694915  -0.888889  -1.834862   
4   64.251252 -350.362905  -1.111111  ... -7.083333  -7.174888   0.623053   

   가동률실적BSI  설비투자실적BSI   채산성실적BSI  자금사정실적BSI  인력사정실적BSI   전기장비ppi  전기장비 제조업  
0       

In [57]:
df_qoq
df_qoq = df_qoq.drop(index=[0, 1, 2,3])


In [58]:
df_qoq.isnull().sum()

,0
index,0
생산지수,0
내수출하지수,0
수출출하지수,0
생산능력지수,0
가동률지수,0
EBITDA,0
인건비,0
합산,0
업황실적BSI,0


In [59]:
df_qoq.to_csv('전기장비qoq.csv')

#데이터YoY변환

In [60]:
import pandas as pd

# df_merged는 원본 데이터프레임
df_yoy = merged.copy()

# 분기 순서대로 정렬
df_yoy = df_yoy.sort_values('index')

# 수치형 컬럼만 선택 (index 제외)
num_cols = df_yoy.select_dtypes(include=['number']).columns

# 전년 동분기 대비 증감률 (%) 계산 (4분기 차이)
df_yoy[num_cols] = df_yoy[num_cols].pct_change(periods=4) * 100

# 결과 확인
print(df_yoy.head(10))


    index      생산지수    내수출하지수     수출출하지수    생산능력지수     가동률지수      EBITDA  \
0  2015Q1       NaN       NaN        NaN       NaN       NaN         NaN   
1  2015Q2       NaN       NaN        NaN       NaN       NaN         NaN   
2  2015Q3       NaN       NaN        NaN       NaN       NaN         NaN   
3  2015Q4       NaN       NaN        NaN       NaN       NaN         NaN   
4  2016Q1  5.780717  0.000000  10.989109  1.108643  3.769748 -266.253707   
5  2016Q2  2.633909 -0.472609   4.890187  0.570002  1.610919  -27.288628   
6  2016Q3  1.369892 -2.235434   4.802724  0.631782  0.482952  -56.377356   
7  2016Q4  2.949394  2.720985   3.427180 -0.922795  3.665540  141.327551   
8  2017Q1  1.203386  4.744360  -4.009243  1.062520  1.246713 -152.233548   
9  2017Q2  6.389907  5.051721   0.867579  2.667661  3.664235   82.318879   

          인건비          합산    업황실적BSI  ...    생산실적BSI  신규수주실적BSI  제품재고실적BSI  \
0         NaN         NaN        NaN  ...        NaN        NaN        NaN   
1      

In [61]:
df_yoy=df_yoy.dropna()

In [62]:
df_yoy

,index,생산지수,내수출하지수,수출출하지수,생산능력지수,가동률지수,EBITDA,인건비,합산,업황실적BSI,...,생산실적BSI,신규수주실적BSI,제품재고실적BSI,가동률실적BSI,설비투자실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,전기장비ppi,전기장비 제조업
4,2016Q1,5.780717,0.000000,10.989109,1.108643,3.769748,-266.253707,309.917311,-265.879603,-12.315271,...,-15.209125,-19.455253,3.525641,-13.671875,-3.180212,-1.550388,-3.906250,0.704225,-1.241225,-8.993165
5,2016Q2,2.633909,-0.472609,4.890187,0.570002,1.610919,-27.288628,-5.477298,-27.276785,-10.900474,...,-9.055118,-14.960630,3.125000,-7.480315,-2.621723,-3.787879,1.171875,4.794521,-0.900901,-10.174518
6,2016Q3,1.369892,-2.235434,4.802724,0.631782,0.482952,-56.377356,-14.720764,-56.359090,4.444444,...,-5.084746,-8.444444,-1.223242,-5.106383,-1.818182,2.845528,5.577689,3.424658,-1.423014,-5.544499
7,2016Q4,2.949394,2.720985,3.427180,-0.922795,3.665540,141.327551,66.373633,141.144235,13.888889,...,1.250000,2.242152,-3.115265,9.606987,2.611940,7.563025,6.882591,-1.683502,-0.123546,2.051499
8,2017Q1,1.203386,4.744360,-4.009243,1.062520,1.246713,-152.233548,-78.916618,-152.351186,16.853933,...,17.040359,17.874396,0.000000,15.384615,2.554745,-8.661417,-1.219512,1.398601,0.999279,10.167808
9,2017Q2,6.389907,5.051721,0.867579,2.667661,3.664235,82.318879,-3.292482,82.258456,28.191489,...,21.212121,23.611111,-1.515152,11.914894,10.000000,3.937008,1.544402,1.307190,0.950413,11.228092
10,2017Q3,4.274600,6.624135,1.453405,1.288342,2.169790,167.289393,2.029938,167.147791,31.914894,...,25.892857,24.757282,1.857585,25.112108,8.148148,1.581028,-1.132075,-2.317881,2.222453,14.106597
11,2017Q4,3.189729,-0.451398,5.111648,2.829256,0.418830,14.594280,1.939445,14.572926,7.804878,...,4.938272,6.140351,3.536977,1.593625,0.000000,-2.343750,-6.818182,-2.054795,2.267807,10.274489
12,2018Q1,-0.113442,3.372804,2.634404,1.742446,-1.755615,34.613005,10.580392,34.597476,7.692308,...,-3.065134,0.000000,-1.857585,-3.137255,4.626335,7.758621,2.880658,0.689655,1.723786,5.774511
13,2018Q2,0.016216,7.517720,-0.892165,1.007585,-0.837195,24.645062,35.257714,24.649036,7.883817,...,1.428571,3.370787,-4.307692,8.365019,4.895105,-2.272727,1.140684,0.645161,2.312730,5.917273


In [63]:
df_yoy.to_csv("전기장비yoy.csv")

# 상관계수 보기(qoq)

In [64]:
qoq = pd.read_csv('전기장비qoq.csv')
qoq = qoq.drop(columns=["Unnamed: 0"])

In [65]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# GDP_QoQ 기준
target = '전기장비 제조업'


In [66]:
# 수치형 컬럼만 선택
num_cols = qoq.select_dtypes(include=['number']).columns

# GDP 기준 상관계수 계산
corr = qoq[num_cols].corr()[target].sort_values(ascending=False)

print("📊 GDP 대비 단순 상관계수")
print(corr)


📊 GDP 대비 단순 상관계수
전기장비 제조업      1.000000
매출실적BSI       0.369143
내수판매실적BSI     0.362965
설비투자실적BSI     0.351571
업황실적BSI       0.336849
수출실적BSI       0.332228
생산실적BSI       0.306155
가동률실적BSI      0.284930
수출출하지수        0.273059
신규수주실적BSI     0.251198
생산능력지수        0.235569
생산지수          0.219165
내수출하지수        0.185221
자금사정실적BSI     0.167014
가동률지수         0.129159
인건비           0.092648
전기장비ppi       0.041012
합산            0.017956
EBITDA        0.017661
채산성실적BSI     -0.073892
인력사정실적BSI    -0.169678
제품재고실적BSI    -0.202048
Name: 전기장비 제조업, dtype: float64


In [67]:
# 시차상관 계산 함수
def lag_corr(x, y, max_lag=4):
    result = {}
    for lag in range(-max_lag, max_lag+1):
        corr = x.corr(y.shift(lag))
        result[lag] = corr
    return pd.Series(result)

# 모든 변수와 GDP 간 시차상관 계산
lags = {}
max_lag = 4  # ±4분기 범위에서 시차 비교

for col in num_cols:
    if col != target:
        lags[col] = lag_corr(qoq[target], qoq[col], max_lag=max_lag)

# 시차상관 결과를 DataFrame으로 정리
df_lagcorr = pd.DataFrame(lags)

print("📈 GDP 기준 시차상관계수")
print(df_lagcorr)
df_lagcorr

📈 GDP 기준 시차상관계수
        생산지수    내수출하지수    수출출하지수    생산능력지수     가동률지수    EBITDA       인건비  \
-4  0.024668  0.237938  0.131986  0.211000 -0.056014 -0.067395  0.134957   
-3  0.149694 -0.193620  0.176139 -0.085445  0.234369  0.031174  0.032341   
-2 -0.100972  0.168579 -0.069058 -0.369179  0.139724  0.100795 -0.033381   
-1 -0.042437 -0.099928  0.088371  0.034957 -0.146438  0.140810  0.105336   
 0  0.219165  0.185221  0.273059  0.235569  0.129159  0.017661  0.092648   
 1  0.333987  0.238888  0.322612  0.118224  0.260471  0.238348  0.048096   
 2  0.137404  0.306401 -0.087113  0.049928  0.190171 -0.092729 -0.109056   
 3 -0.451634 -0.119108 -0.240829 -0.363663 -0.332738 -0.097250  0.046067   
 4  0.105178 -0.219852 -0.000933  0.033944  0.124331 -0.216133  0.151422   

          합산   업황실적BSI   매출실적BSI  ...  내수판매실적BSI   생산실적BSI  신규수주실적BSI  \
-4 -0.066532 -0.149412 -0.205658  ...  -0.229259 -0.244657  -0.242770   
-3  0.031629 -0.003462  0.034612  ...   0.010064  0.044584   0.037610   
-2  

,생산지수,내수출하지수,수출출하지수,생산능력지수,가동률지수,EBITDA,인건비,합산,업황실적BSI,매출실적BSI,...,내수판매실적BSI,생산실적BSI,신규수주실적BSI,제품재고실적BSI,가동률실적BSI,설비투자실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,전기장비ppi
-4,0.024668,0.237938,0.131986,0.211000,-0.056014,-0.067395,0.134957,-0.066532,-0.149412,-0.205658,...,-0.229259,-0.244657,-0.242770,-0.060032,-0.247780,-0.193711,-0.176150,-0.230192,0.084020,-0.077633
-3,0.149694,-0.193620,0.176139,-0.085445,0.234369,0.031174,0.032341,0.031629,-0.003462,0.034612,...,0.010064,0.044584,0.037610,0.192598,0.125930,-0.080653,0.141544,0.090600,0.066877,0.210852
-2,-0.100972,0.168579,-0.069058,-0.369179,0.139724,0.100795,-0.033381,0.100785,0.003464,0.088106,...,0.080579,-0.037319,0.053496,0.187403,-0.109943,0.177991,-0.089914,-0.130564,-0.312681,0.105326
-1,-0.042437,-0.099928,0.088371,0.034957,-0.146438,0.140810,0.105336,0.140552,-0.004896,0.091041,...,0.175636,0.096989,0.167319,-0.011855,0.000397,0.236524,0.025657,-0.048440,-0.133048,0.060217
0,0.219165,0.185221,0.273059,0.235569,0.129159,0.017661,0.092648,0.017956,0.336849,0.369143,...,0.362965,0.306155,0.251198,-0.202048,0.284930,0.351571,-0.073892,0.167014,-0.169678,0.041012
1,0.333987,0.238888,0.322612,0.118224,0.260471,0.238348,0.048096,0.238731,0.171768,0.088294,...,0.088503,0.201029,0.100123,-0.188955,0.251859,-0.131975,0.226243,0.139357,0.133133,-0.071356
2,0.137404,0.306401,-0.087113,0.049928,0.190171,-0.092729,-0.109056,-0.092774,-0.023972,-0.004095,...,-0.033230,-0.035830,-0.057393,-0.114842,-0.039747,-0.017036,-0.143146,-0.031370,0.012148,-0.127892
3,-0.451634,-0.119108,-0.240829,-0.363663,-0.332738,-0.097250,0.046067,-0.097329,-0.010921,-0.060848,...,-0.051784,-0.088628,-0.088480,0.173225,-0.080147,0.078739,0.097525,0.018015,-0.026602,-0.208387
4,0.105178,-0.219852,-0.000933,0.033944,0.124331,-0.216133,0.151422,-0.215800,-0.044361,0.011675,...,0.072884,-0.074485,-0.001387,-0.093895,-0.083637,0.043781,0.018135,-0.016733,-0.060671,-0.216299


In [68]:
# 동행(0) + 후행(+1~+4) 구간만 추출
df_lagcorr_follow = df_lagcorr.loc[0:4]

# 각 변수별로 절댓값 기준 최대 상관계수와 그때의 lag 찾기
max_corr_lag = df_lagcorr_follow.apply(lambda x: x.abs().idxmax())
max_corr_value = df_lagcorr_follow.apply(lambda x: x.loc[x.abs().idxmax()])

# 결과 DataFrame으로 정리
result_follow = pd.DataFrame({
    '최대상관계수': max_corr_value,
    '시차(lag)': max_corr_lag
}).sort_values(by='최대상관계수', ascending=False)

# 상위 10개 출력
print("📊 GDP 기준 동행·후행 변수 TOP 10")
print(result_follow.head(10))


📊 GDP 기준 동행·후행 변수 TOP 10
              최대상관계수  시차(lag)
매출실적BSI     0.369143        0
내수판매실적BSI   0.362965        0
설비투자실적BSI   0.351571        0
업황실적BSI     0.336849        0
수출실적BSI     0.332228        0
수출출하지수      0.322612        1
내수출하지수      0.306401        2
생산실적BSI     0.306155        0
가동률실적BSI    0.284930        0
신규수주실적BSI   0.251198        0


In [69]:
# df2 사본
df2 = qoq.copy()

# target 및 feature 지정
target_col = '전기장비 제조업'
feature_cols = [col for col in df2.columns if col not in ['index', target_col]]

# 결과 저장용 데이터프레임
corr_df = pd.DataFrame(columns=['Feature', 'Lag', 'Correlation'])

# lag 1~4 생성 및 상관계수 계산
for col in feature_cols:
    for lag in range(1, 5):  # lag 1~4
        lag_col_name = f"{col}_lag{lag}"
        df2[lag_col_name] = df2[col].shift(lag)  # 실제 데이터에 lag 컬럼 추가
        corr = df2[target_col].corr(df2[lag_col_name])  # 상관계수 계산
        corr_df = pd.concat(
            [corr_df, pd.DataFrame({'Feature': [col], 'Lag': [lag], 'Correlation': [corr]})],
            ignore_index=True
        )

# 상관계수 높은 순으로 정렬
corr_df.sort_values(by='Correlation', ascending=False, inplace=True)
corr_df.reset_index(drop=True, inplace=True)

corr_df


/tmp/ipython-input-1274427553.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  corr_df = pd.concat(


,Feature,Lag,Correlation
0,생산지수,1,0.333987
1,수출출하지수,1,0.322612
2,내수출하지수,2,0.306401
3,가동률지수,1,0.260471
4,가동률실적BSI,1,0.251859
...,...,...,...
79,내수출하지수,4,-0.219852
80,수출출하지수,3,-0.240829
81,가동률지수,3,-0.332738
82,생산능력지수,3,-0.363663


In [70]:
# 숫자형 컬럼만 선택 (연도분기 제외)
numeric_df = df2.select_dtypes(include=["number"])

# 전체 상관계수
corr = numeric_df.corr()

# 광업전기대비증감률과 다른 컬럼들 상관계수만 보기
target_corr = corr["전기장비 제조업"].drop("전기장비 제조업")
target_corr = target_corr.sort_values(ascending=False)
target_corr

,전기장비 제조업
매출실적BSI,0.369143
내수판매실적BSI,0.362965
설비투자실적BSI,0.351571
업황실적BSI,0.336849
생산지수_lag1,0.333987
...,...
내수출하지수_lag4,-0.219852
수출출하지수_lag3,-0.240829
가동률지수_lag3,-0.332738
생산능력지수_lag3,-0.363663


In [71]:
target_corr.to_csv('전기장비순서.csv')

In [72]:
df2.dropna(inplace=True)

In [73]:
df2

,index,생산지수,내수출하지수,수출출하지수,생산능력지수,가동률지수,EBITDA,인건비,합산,업황실적BSI,...,자금사정실적BSI_lag3,자금사정실적BSI_lag4,인력사정실적BSI_lag1,인력사정실적BSI_lag2,인력사정실적BSI_lag3,인력사정실적BSI_lag4,전기장비ppi_lag1,전기장비ppi_lag2,전기장비ppi_lag3,전기장비ppi_lag4
4,2017Q1,-0.323804,0.549112,-1.798031,1.231221,-0.780722,-45.590800,-79.185573,-45.647488,1.463415,...,5.284553,-0.404858,-3.311258,-1.307190,6.993007,-3.703704,0.747741,-0.526860,-0.278150,-0.061773
5,2017Q2,1.753192,-0.448933,0.802409,1.216246,-0.086565,67.399237,-3.012489,67.353738,15.865385,...,2.316602,5.284553,-0.684932,-3.311258,-1.307190,6.993007,1.061746,0.747741,-0.526860,-0.278150
6,2017Q3,-0.288853,2.322694,2.327072,-0.389129,0.157704,13.199618,-1.084406,13.194269,2.904564,...,-0.377358,2.316602,6.896552,-0.684932,-3.311258,-1.307190,-0.326397,1.061746,0.747741,-0.526860
7,2017Q4,2.035962,-2.806093,3.769372,0.750047,1.137181,11.145572,410.501734,11.276261,-10.887097,...,-7.954545,-0.377358,-4.838710,6.896552,-0.684932,-3.311258,0.726566,-0.326397,1.061746,0.747741
8,2018Q1,-3.514504,4.411749,-4.112430,0.161300,-2.929192,-36.085938,-77.421227,-36.147996,1.357466,...,8.230453,-7.954545,-3.050847,-4.838710,6.896552,-0.684932,0.792441,0.726566,-0.326397,1.061746
9,2018Q2,1.885273,3.542744,-2.661211,0.485186,0.847458,55.003510,18.631421,54.984201,16.071429,...,-0.380228,8.230453,2.097902,-3.050847,-4.838710,6.896552,0.524141,0.792441,0.726566,-0.326397
10,2018Q3,-0.377982,-0.419426,2.990570,0.321553,-1.355729,13.618283,5.729783,13.615078,-11.538462,...,-6.106870,-0.380228,6.849315,2.097902,-3.050847,-4.838710,0.250677,0.524141,0.792441,0.726566
11,2018Q4,1.913355,-0.181361,-5.089381,0.544786,1.768458,23.226490,425.557588,23.378635,-0.434783,...,1.626016,-6.106870,0.320513,6.849315,2.097902,-3.050847,-0.650130,0.250677,0.524141,0.792441
12,2019Q1,-2.738796,0.393165,10.134457,-0.159902,-2.494622,-28.008346,-81.059352,-28.093803,-6.113537,...,6.400000,1.626016,-4.792332,0.320513,6.849315,2.097902,0.070472,-0.650130,0.250677,0.524141
13,2019Q2,2.939064,-0.060326,3.286599,-0.160158,2.395972,4.617247,-4.132954,4.613534,4.186047,...,-6.390977,6.400000,1.677852,-4.792332,0.320513,6.849315,0.301811,0.070472,-0.650130,0.250677


In [74]:
df2 = df2.reset_index()
df2 = df2.drop(columns=["level_0"])


In [75]:
df2.to_csv('전기장비lag.csv', encoding='utf-8-sig')

In [76]:
print(df2.columns)


Index(['index', '생산지수', '내수출하지수', '수출출하지수', '생산능력지수', '가동률지수', 'EBITDA', '인건비',
       '합산', '업황실적BSI',
       ...
       '자금사정실적BSI_lag3', '자금사정실적BSI_lag4', '인력사정실적BSI_lag1', '인력사정실적BSI_lag2',
       '인력사정실적BSI_lag3', '인력사정실적BSI_lag4', '전기장비ppi_lag1', '전기장비ppi_lag2',
       '전기장비ppi_lag3', '전기장비ppi_lag4'],
      dtype='object', length=107)


#ARIMA

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('전기장비lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = '전기장비 제조업'   # GDP 예측 대상 칼럼
exog_vars = [
    '매출실적BSI', '내수판매실적BSI', '설비투자실적BSI '
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2017Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (3.0, 1.0, 2.0)
📉 평균 Train MAE: 2.2601
📈 평균 Test MAE: 2.5059


## ARIMA 예측값

In [77]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('전기장비lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = '전기장비 제조업'
exog_vars = ['매출실적BSI', '내수판매실적BSI', '설비투자실적BSI ']

y = df[target].dropna()
X = df[exog_vars].dropna()

df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2017Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    pred = fit.forecast(steps=1, exog=X_test)

                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 최적 (p,d,q) 선택
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

best_p, best_d, best_q = int(best['p']), int(best['d']), int(best['q'])

print("✅ 최적 ARIMAX(p,d,q):", (best_p, best_d, best_q))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

# ---------------------------------------------------
# 8️⃣ 최적 (p,d,q)로 전체 구간 롤링 예측값 저장
# ---------------------------------------------------
rolling_preds = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    train_data = y.loc[train_start:train_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(train_data) < train_window or len(X_test) == 0:
        continue

    try:
        model = SARIMAX(train_data, exog=X_train, order=(best_p, best_d, best_q),
                        enforce_stationarity=False, enforce_invertibility=False)
        fit = model.fit(disp=False)
        pred = fit.forecast(steps=1, exog=X_test)

        rolling_preds.append({
            'date': test_end,
            'pred': float(pred),
            'actual': float(y.loc[test_end])
        })
    except:
        continue

pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 최적 모델 예측값(pred) 시계열:")
print(pred_df)


✅ 최적 ARIMAX(p,d,q): (3, 1, 2)
📉 평균 Train MAE: 2.2601
📈 평균 Test MAE: 2.5059

📌 최적 모델 예측값(pred) 시계열:
            pred    actual
date                      
2022Q1  7.723651  5.842324
2022Q2 -0.449448  0.267532
2022Q3 -3.890779 -3.314667
2022Q4 -1.061917 -3.390126
2023Q1  0.561319 -0.167099
2023Q2  2.521847  1.198797
2023Q3  0.524786  4.477996
2023Q4  2.370049 -4.657232
2024Q1 -2.231566 -2.547821
2024Q2 -2.220798 -0.735633
2024Q3  1.909485  2.741664
2024Q4  0.837295 -4.554742
2025Q1 -2.510679 -1.866263
2025Q2 -5.165492  2.712833


In [80]:
level_2025Q1 = 8297.6

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 7868.988153757635


# 랜덤포레스트

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기장비lag.csv')

# 인덱스 처리 (index 컬럼 없을 경우 자동 생성)
if 'index' not in df.columns:
    df.index = pd.period_range(start='2017Q1', periods=len(df), freq='Q')
else:
    df['index'] = pd.PeriodIndex(df['index'], freq='Q')
    df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = '전기장비 제조업'
exog_vars = ['매출실적BSI', '내수판매실적BSI', '설비투자실적BSI ']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y, X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2017Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 결과 저장
# -----------------------------
train_mae_list = []
test_mae_list = []

# -----------------------------
# 5️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 랜덤포레스트 모델 학습
        rf = RandomForestRegressor(
            n_estimators=500,     # 트리 개수
            max_depth=None,      # 깊이 제한 없음
            random_state=42
        )
        rf.fit(X_train, y_train)

        # 학습 구간 예측 (Train MAE)
        train_pred = rf.predict(X_train)
        train_mae = mean_absolute_error(y_train, train_pred)
        train_mae_list.append(train_mae)

        # 테스트 구간 예측 (Test MAE)
        test_pred = rf.predict(X_test)
        test_mae = mean_absolute_error(y_test, test_pred)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약 출력
# -----------------------------
print("✅ 랜덤포레스트 롤링 예측 결과")
print("📉 평균 Train MAE:", round(np.mean(train_mae_list), 4))
print("📈 평균 Test MAE:", round(np.mean(test_mae_list), 4))


✅ 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 1.3496
📈 평균 Test MAE: 3.7894


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기장비lag.csv')

# 인덱스 처리 (index 컬럼 없을 경우 자동 생성)
if 'index' not in df.columns:
    df.index = pd.period_range(start='2017Q1', periods=len(df), freq='Q')
else:
    df['index'] = pd.PeriodIndex(df['index'], freq='Q')
    df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = '전기장비 제조업'
exog_vars = ['매출실적BSI', '내수판매실적BSI', '설비투자실적BSI ', '업황실적BSI']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y, X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2017Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 결과 저장
# -----------------------------
train_mae_list = []
test_mae_list = []

# -----------------------------
# 5️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 랜덤포레스트 모델 학습
        rf = RandomForestRegressor(
            n_estimators=500,     # 트리 개수
            max_depth=None,      # 깊이 제한 없음
            random_state=42
        )
        rf.fit(X_train, y_train)

        # 학습 구간 예측 (Train MAE)
        train_pred = rf.predict(X_train)
        train_mae = mean_absolute_error(y_train, train_pred)
        train_mae_list.append(train_mae)

        # 테스트 구간 예측 (Test MAE)
        test_pred = rf.predict(X_test)
        test_mae = mean_absolute_error(y_test, test_pred)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약 출력
# -----------------------------
print("✅ 랜덤포레스트 롤링 예측 결과")
print("📉 평균 Train MAE:", round(np.mean(train_mae_list), 4))
print("📈 평균 Test MAE:", round(np.mean(test_mae_list), 4))


✅ 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 1.3278
📈 평균 Test MAE: 3.7153


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기장비lag.csv')

# 인덱스 처리 (index 컬럼 없을 경우 자동 생성)
if 'index' not in df.columns:
    df.index = pd.period_range(start='2017Q1', periods=len(df), freq='Q')
else:
    df['index'] = pd.PeriodIndex(df['index'], freq='Q')
    df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = '전기장비 제조업'
exog_vars = ['매출실적BSI', '내수판매실적BSI', '설비투자실적BSI ', '업황실적BSI', '생산지수_lag3','생산능력지수_lag3','가동률지수_lag3'
    ]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y, X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2017Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 결과 저장
# -----------------------------
train_mae_list = []
test_mae_list = []

# -----------------------------
# 5️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 랜덤포레스트 모델 학습
        rf = RandomForestRegressor(
            n_estimators=500,     # 트리 개수
            max_depth=None,      # 깊이 제한 없음
            random_state=42
        )
        rf.fit(X_train, y_train)

        # 학습 구간 예측 (Train MAE)
        train_pred = rf.predict(X_train)
        train_mae = mean_absolute_error(y_train, train_pred)
        train_mae_list.append(train_mae)

        # 테스트 구간 예측 (Test MAE)
        test_pred = rf.predict(X_test)
        test_mae = mean_absolute_error(y_test, test_pred)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약 출력
# -----------------------------
print("✅ 랜덤포레스트 롤링 예측 결과")
print("📉 평균 Train MAE:", round(np.mean(train_mae_list), 4))
print("📈 평균 Test MAE:", round(np.mean(test_mae_list), 4))


✅ 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 1.2311
📈 평균 Test MAE: 3.3029


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기장비lag.csv')

# 인덱스 처리 (index 컬럼 없을 경우 자동 생성)
if 'index' not in df.columns:
    df.index = pd.period_range(start='2017Q1', periods=len(df), freq='Q')
else:
    df['index'] = pd.PeriodIndex(df['index'], freq='Q')
    df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = '전기장비 제조업'
exog_vars = ['매출실적BSI', '내수판매실적BSI', '설비투자실적BSI ',  '생산지수_lag3','생산능력지수_lag3','가동률지수_lag3'
    ]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y, X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2017Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 결과 저장
# -----------------------------
train_mae_list = []
test_mae_list = []

# -----------------------------
# 5️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 랜덤포레스트 모델 학습
        rf = RandomForestRegressor(
            n_estimators=500,     # 트리 개수
            max_depth=None,      # 깊이 제한 없음
            random_state=42
        )
        rf.fit(X_train, y_train)

        # 학습 구간 예측 (Train MAE)
        train_pred = rf.predict(X_train)
        train_mae = mean_absolute_error(y_train, train_pred)
        train_mae_list.append(train_mae)

        # 테스트 구간 예측 (Test MAE)
        test_pred = rf.predict(X_test)
        test_mae = mean_absolute_error(y_test, test_pred)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약 출력
# -----------------------------
print("✅ 랜덤포레스트 롤링 예측 결과")
print("📉 평균 Train MAE:", round(np.mean(train_mae_list), 4))
print("📈 평균 Test MAE:", round(np.mean(test_mae_list), 4))


✅ 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 1.2357
📈 평균 Test MAE: 3.0444


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기장비lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = '전기장비 제조업'
exog_vars = ['매출실적BSI', '내수판매실적BSI', '설비투자실적BSI ',  '생산지수_lag3','생산능력지수_lag3','가동률지수_lag3'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2017Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [8, 10, 12],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt']
}

# 첫 번째 윈도우 구간 설정
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

# 하이퍼파라미터 탐색 (단 1회)
rf = RandomForestRegressor(random_state=42)
rand_search = RandomizedSearchCV(
    rf,
    param_distributions=param_grid,
    n_iter=10,            # 108조합 중 10개만 탐색
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 고정된 최적 파라미터로 학습
        model = RandomForestRegressor(**best_params, random_state=42)
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE 계산
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 랜덤포레스트 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'n_estimators': 400, 'min_samples_split': 5, 'max_features': 'sqrt', 'max_depth': 12}

📊 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 1.6646
📈 평균 Test MAE: 3.0959


## 랜포 예측값

In [81]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기장비lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ y, X 설정
# -----------------------------
target = '전기장비 제조업'
exog_vars = [
    '매출실적BSI', '내수판매실적BSI', '설비투자실적BSI ',
    '생산지수_lag3','생산능력지수_lag3','가동률지수_lag3'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2017Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ RandomizedSearchCV (초기 구간 1회)
# -----------------------------
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [8, 10, 12],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt']
}

first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

rf = RandomForestRegressor(random_state=42)
rand_search = RandomizedSearchCV(
    rf,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 + pred 저장
# -----------------------------
train_mae_list = []
test_mae_list = []
rolling_preds = []   # ⭐ 예측값 저장 리스트

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = RandomForestRegressor(**best_params, random_state=42)
        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(test_pred),
            'actual': float(y_test.values[0])
        })

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 정리
# -----------------------------
print("\n📊 랜덤포레스트 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")

pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 랜덤포레스트 예측값(pred) 시계열:")
print(pred_df)


✅ 최적 하이퍼파라미터: {'n_estimators': 400, 'min_samples_split': 5, 'max_features': 'sqrt', 'max_depth': 12}

📊 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 1.6646
📈 평균 Test MAE: 3.0959

📌 랜덤포레스트 예측값(pred) 시계열:
            pred    actual
date                      
2022Q1  2.663988  5.842324
2022Q2  1.896033  0.267532
2022Q3  1.194283 -3.314667
2022Q4  2.469412 -3.390126
2023Q1  0.024151 -0.167099
2023Q2 -0.619416  1.198797
2023Q3  3.018392  4.477996
2023Q4 -1.516479 -4.657232
2024Q1  2.589457 -2.547821
2024Q2  1.194735 -0.735633
2024Q3 -0.772726  2.741664
2024Q4  0.122523 -4.554742
2025Q1  0.632939 -1.866263
2025Q2 -1.086328  2.712833


In [82]:
level_2025Q1 = 8297.6

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 8207.460874902903


# AR

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기장비qoq.csv')

# GDP 단일 시계열만 사용 (AR(1))
y = df['전기장비 제조업'].dropna()
y.index = pd.PeriodIndex(df.loc[y.index, 'index'], freq='Q')

# -----------------------------
# 2️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

train_mae_list = []
test_mae_list = []

# -----------------------------
# 3️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    train_data = y.loc[train_start:train_end]
    test_data = y.loc[test_end:test_end]

    if len(train_data) < train_window:
        continue

    try:
        # AR(1) 학습
        model = AutoReg(train_data, lags=1, old_names=False)
        fit = model.fit()

        # --- Train 예측 ---
        # fittedvalues는 길이가 n-1이므로 첫 번째 값 제외하고 MAE 계산
        train_mae = mean_absolute_error(train_data[1:], fit.fittedvalues)
        train_mae_list.append(train_mae)

        # --- Test 예측 (1분기 앞 예측) ---
        pred = fit.predict(start=len(train_data), end=len(train_data))
        test_mae = mean_absolute_error(test_data, pred)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 4️⃣ 결과 출력
# -----------------------------
print("✅ AR(1) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ AR(1) 롤링 예측 결과
📉 평균 Train MAE: 3.0799
📈 평균 Test MAE: 2.9676


## AR 예측값

In [83]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기장비qoq.csv')

# GDP 단일 시계열
y = df['전기장비 제조업'].dropna()
y.index = pd.PeriodIndex(df.loc[y.index, 'index'], freq='Q')

# -----------------------------
# 2️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

train_mae_list = []
test_mae_list = []

# ⭐ 예측값 저장할 리스트
rolling_preds = []

# -----------------------------
# 3️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    train_data = y.loc[train_start:train_end]
    test_data = y.loc[test_end:test_end]

    if len(train_data) < train_window:
        continue

    try:
        # AR(1) 학습
        model = AutoReg(train_data, lags=1, old_names=False)
        fit = model.fit()

        # --- Train MAE ---
        train_mae = mean_absolute_error(train_data[1:], fit.fittedvalues)
        train_mae_list.append(train_mae)

        # --- Test 예측 ---
        pred = fit.predict(start=len(train_data), end=len(train_data))
        test_mae = mean_absolute_error(test_data, pred)
        test_mae_list.append(test_mae)

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(pred),
            'actual': float(test_data.values[0])
        })

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 4️⃣ 결과 출력
# -----------------------------
print("✅ AR(1) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")

# -----------------------------
# 5️⃣ pred 시계열 데이터프레임 출력
# -----------------------------
pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 AR(1) 예측값(pred) 시계열:")
print(pred_df)


✅ AR(1) 롤링 예측 결과
📉 평균 Train MAE: 3.0799
📈 평균 Test MAE: 2.9676

📌 AR(1) 예측값(pred) 시계열:
            pred    actual
date                      
2021Q1 -0.771004  2.907648
2021Q2  1.232106 -0.130390
2021Q3  1.963444 -2.846211
2021Q4  2.348448  4.408998
2022Q1  0.357684  5.842324
2022Q2  0.556166  0.267532
2022Q3  1.371332 -3.314667
2022Q4  1.587722 -3.390126
2023Q1  0.649860 -0.167099
2023Q2  0.721498  1.198797
2023Q3  0.785374  4.477996
2023Q4  0.913790 -4.657232
2024Q1  0.408815 -2.547821
2024Q2 -0.232855 -0.735633
2024Q3  0.234061  2.741664
2024Q4  1.164140 -4.554742
2025Q1 -0.738577 -1.866263
2025Q2  0.016359  2.712833


In [84]:
level_2025Q1 = 8297.6

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 8298.95739670467


#XGB

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기장비lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = '전기장비 제조업'
exog_vars = [
    '매출실적BSI', '내수판매실적BSI', '설비투자실적BSI ',
    '생산지수_lag3', '생산능력지수_lag3', '가동률지수_lag3'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2017Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 하이퍼파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'n_estimators': [200, 400, 600],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3]
}

# 첫 번째 윈도우로 탐색
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

# RandomizedSearchCV 실행
xgb = XGBRegressor(random_state=42, objective='reg:squarederror')
rand_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_grid,
    n_iter=15,          # 전체 중 15조합 탐색
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 고정된 최적 파라미터로 학습
        model = XGBRegressor(**best_params, random_state=42, objective='reg:squarederror')
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE 계산
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 XGBoost 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'subsample': 0.8, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 5, 'learning_rate': 0.03, 'gamma': 0.3, 'colsample_bytree': 0.8}

📊 XGBoost 롤링 예측 결과
📉 평균 Train MAE: 1.4020
📈 평균 Test MAE: 3.1500


##XGB 예측값

In [85]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기장비lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ y, X 설정
# -----------------------------
target = '전기장비 제조업'
exog_vars = [
    '매출실적BSI', '내수판매실적BSI', '설비투자실적BSI ',
    '생산지수_lag3', '생산능력지수_lag3', '가동률지수_lag3'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2017Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ RandomizedSearchCV (초기 구간)
# -----------------------------
param_grid = {
    'n_estimators': [200, 400, 600],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3]
}

first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

xgb = XGBRegressor(random_state=42, objective='reg:squarederror')
rand_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_grid,
    n_iter=15,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 + pred 저장
# -----------------------------
train_mae_list = []
test_mae_list = []
rolling_preds = []   # ⭐ 예측값 저장

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = XGBRegressor(**best_params, random_state=42, objective='reg:squarederror')
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(test_pred),
            'actual': float(y_test.values[0])
        })

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 XGBoost 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")

pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 XGBoost 예측값(pred) 시계열:")
print(pred_df)


✅ 최적 하이퍼파라미터: {'subsample': 0.8, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 5, 'learning_rate': 0.03, 'gamma': 0.3, 'colsample_bytree': 0.8}

📊 XGBoost 롤링 예측 결과
📉 평균 Train MAE: 1.4020
📈 평균 Test MAE: 3.1500

📌 XGBoost 예측값(pred) 시계열:
            pred    actual
date                      
2022Q1  1.525646  5.842324
2022Q2  1.315158  0.267532
2022Q3  1.148519 -3.314667
2022Q4  1.520859 -3.390126
2023Q1  1.559204 -0.167099
2023Q2 -1.240709  1.198797
2023Q3  1.805433  4.477996
2023Q4 -1.948887 -4.657232
2024Q1  1.764948 -2.547821
2024Q2  1.092644 -0.735633
2024Q3 -0.786187  2.741664
2024Q4 -0.927528 -4.554742
2025Q1  0.146662 -1.866263
2025Q2 -1.792564  2.712833


In [86]:
level_2025Q1 = 8297.6

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 8148.860196784974


# MLP

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기장비lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = '전기장비 제조업'
exog_vars = [
    '매출실적BSI', '내수판매실적BSI', '설비투자실적BSI ',
    '생산지수_lag3', '생산능력지수_lag3', '가동률지수_lag3'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# 스케일링 (NN에서는 매우 중요)
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(), index=y.index)

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2017Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 하이퍼파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'hidden_layer_sizes': [(32,), (64,), (32,16), (64,32), (128,64)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'lbfgs'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}

# 첫 번째 윈도우로 하이퍼파라미터 탐색
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X_scaled.loc[start_period:first_train_end]
first_y_train = y_scaled.loc[start_period:first_train_end]

mlp = MLPRegressor(random_state=42, max_iter=1000)
rand_search = RandomizedSearchCV(
    mlp,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y_scaled.loc[train_start:train_end]
    y_test = y_scaled.loc[test_end:test_end]
    X_train = X_scaled.loc[train_start:train_end]
    X_test = X_scaled.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = MLPRegressor(**best_params, random_state=42, max_iter=1000)
        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # 스케일링 복원
        train_pred_inv = scaler_y.inverse_transform(train_pred.reshape(-1, 1)).flatten()
        test_pred_inv = scaler_y.inverse_transform(test_pred.reshape(-1, 1)).flatten()
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1, 1)).flatten()
        y_test_inv = scaler_y.inverse_transform(y_test.values.reshape(-1, 1)).flatten()

        # MAE 계산
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 Neural Network (MLP) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'solver': 'adam', 'learning_rate_init': 0.001, 'hidden_layer_sizes': (64,), 'alpha': 0.01, 'activation': 'relu'}

📊 Neural Network (MLP) 롤링 예측 결과
📉 평균 Train MAE: 0.4536
📈 평균 Test MAE: 4.4022


## MLP 예측값

In [91]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기장비lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ y, X 설정
# -----------------------------
target = '전기장비 제조업'
exog_vars = [
    '매출실적BSI', '내수판매실적BSI', '설비투자실적BSI ',
    '생산지수_lag3', '생산능력지수_lag3', '가동률지수_lag3'
]

df_model = df[[target] + exog_vars].dropna()

y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 스케일링 (정상 코드)
# -----------------------------
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(
    scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(),
    index=y.index
)

# -----------------------------
# 4️⃣ 롤링 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2017Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 5️⃣ 하이퍼파라미터 축소 (안정성)
# -----------------------------
param_grid = {
    'hidden_layer_sizes': [(32,), (64,), (32,16)],
    'activation': ['relu'],
    'solver': ['adam'],
    'alpha': [0.0001, 0.001],
    'learning_rate_init': [0.001, 0.005]
}

first_train_end = start_period + (train_window - 1)
first_X_train = X_scaled.loc[start_period:first_train_end]
first_y_train = y_scaled.loc[start_period:first_train_end]

mlp = MLPRegressor(random_state=42, max_iter=2000)
rand_search = RandomizedSearchCV(
    mlp, param_grid, n_iter=5, cv=3,
    scoring='neg_mean_absolute_error', n_jobs=-1, random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 파라미터:", best_params)

# -----------------------------
# 6️⃣ 롤링 예측
# -----------------------------
train_mae_list = []
test_mae_list = []
rolling_preds = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    train_start = train_end - (train_window - 1)
    test_end = test_start

    X_train = X_scaled.loc[train_start:train_end]
    y_train = y_scaled.loc[train_start:train_end]
    X_test = X_scaled.loc[test_end:test_end]
    y_test = y_scaled.loc[test_end:test_end]

    if len(X_train) < train_window:
        continue

    try:
        model = MLPRegressor(
            **best_params,
            random_state=42,
            max_iter=2000
        )
        model.fit(X_train, y_train)

        # ---- Train 예측 ----
        train_pred_scaled = model.predict(X_train)
        train_pred = scaler_y.inverse_transform(train_pred_scaled.reshape(-1,1)).flatten()
        train_real = scaler_y.inverse_transform(y_train.values.reshape(-1,1)).flatten()
        train_mae = mean_absolute_error(train_real, train_pred)
        train_mae_list.append(train_mae)

        # ---- Test 예측 ----
        test_pred_scaled = model.predict(X_test)
        test_pred = scaler_y.inverse_transform(test_pred_scaled.reshape(-1,1)).flatten()
        test_real = scaler_y.inverse_transform(y_test.values.reshape(-1,1)).flatten()
        test_mae = mean_absolute_error(test_real, test_pred)
        test_mae_list.append(test_mae)

        # ---- pred 저장 ----
        rolling_preds.append({
            'date': test_end,
            'pred': float(test_pred[0]),
            'actual': float(test_real[0])
        })

    except Exception as e:
        print("❌ 오류 발생:", e)
        continue

# -----------------------------
# 7️⃣ 결과 출력
# -----------------------------
print("\n📊 Neural Network (MLP) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")

if len(rolling_preds) > 0:
    pred_df = pd.DataFrame(rolling_preds).set_index('date')
    print(pred_df)
else:
    print("⚠ rolling_preds가 텅 빔 → 모델 학습 실패")


✅ 최적 파라미터: {'solver': 'adam', 'learning_rate_init': 0.001, 'hidden_layer_sizes': (64,), 'alpha': 0.001, 'activation': 'relu'}

📊 Neural Network (MLP) 롤링 예측 결과
📉 평균 Train MAE: 0.4241
📈 평균 Test MAE: 4.4846
             pred    actual
date                       
2022Q1   1.065699  5.842324
2022Q2   3.072443  0.267532
2022Q3   0.528665 -3.314667
2022Q4   6.197137 -3.390126
2023Q1   0.371793 -0.167099
2023Q2   2.139057  1.198797
2023Q3  11.410291  4.477996
2023Q4 -13.759665 -4.657232
2024Q1   0.493672 -2.547821
2024Q2   6.188217 -0.735633
2024Q3  -1.440296  2.741664
2024Q4   2.737262 -4.554742
2025Q1  -0.879373 -1.866263
2025Q2   0.880723  2.712833


In [92]:
level_2025Q1 = 8297.6

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 8370.678909590544


#LSTM

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기장비lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = '전기장비 제조업'
exog_vars = [
    '매출실적BSI', '내수판매실적BSI', '설비투자실적BSI ',
    '생산지수_lag3', '생산능력지수_lag3', '가동률지수_lag3'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 스케일링
# -----------------------------
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(), index=y.index)

# -----------------------------
# 4️⃣ LSTM 입력 형태 변환 함수
# -----------------------------
def create_lstm_input(X, y, lookback=4):
    """
    lookback: 과거 몇 분기 데이터를 사용할지 (ex. 4 = 1년)
    """
    Xs, ys, idxs = [], [], []
    for i in range(lookback, len(X)):
        Xs.append(X.iloc[i-lookback:i].values)
        ys.append(y.iloc[i])
        idxs.append(y.index[i])
    return np.array(Xs), np.array(ys), idxs

lookback = 4
X_seq, y_seq, idxs = create_lstm_input(X_scaled, y_scaled, lookback)
y_seq = pd.Series(y_seq, index=idxs)

# -----------------------------
# 5️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2017Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 6️⃣ LSTM 모델 정의 함수
# -----------------------------
def build_lstm(input_shape):
    model = Sequential()
    model.add(LSTM(64, activation='tanh', input_shape=input_shape, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mae')
    return model

# -----------------------------
# 7️⃣ 롤링 예측
# -----------------------------
train_mae_list, test_mae_list = [], []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    # 윈도우 범위 맞추기
    train_mask = (np.array(idxs) >= train_start) & (np.array(idxs) <= train_end)
    test_mask = (np.array(idxs) == test_end)

    X_train, y_train = X_seq[train_mask], y_seq[train_mask]
    X_test, y_test = X_seq[test_mask], y_seq[test_mask]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = build_lstm((X_train.shape[1], X_train.shape[2]))
        early_stop = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)
        model.fit(X_train, y_train, epochs=200, batch_size=8, verbose=0, callbacks=[early_stop])

        # 예측
        train_pred = model.predict(X_train, verbose=0)
        test_pred = model.predict(X_test, verbose=0)

        # 스케일링 복원
        train_pred_inv = scaler_y.inverse_transform(train_pred)
        test_pred_inv = scaler_y.inverse_transform(test_pred)
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1, 1))
        y_test_inv = scaler_y.inverse_transform(y_test.values.reshape(-1, 1))

        # MAE 계산
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 8️⃣ 결과 요약
# -----------------------------
print("\n📊 LSTM 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")



📊 LSTM 롤링 예측 결과
📉 평균 Train MAE: 0.7800
📈 평균 Test MAE: 2.7868


## lstm 예측값

In [93]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기장비lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ y, X 설정
# -----------------------------
target = '전기장비 제조업'
exog_vars = [
    '매출실적BSI', '내수판매실적BSI', '설비투자실적BSI ',
    '생산지수_lag3', '생산능력지수_lag3', '가동률지수_lag3'
]

df_model = pd.concat([df[target], df[exog_vars]], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 스케일링
# -----------------------------
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(
    scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(),
    index=y.index
)

# -----------------------------
# 4️⃣ LSTM 입력 생성
# -----------------------------
def create_lstm_input(X, y, lookback=4):
    Xs, ys, idxs = [], [], []
    for i in range(lookback, len(X)):
        Xs.append(X.iloc[i-lookback:i].values)
        ys.append(y.iloc[i])
        idxs.append(y.index[i])
    return np.array(Xs), np.array(ys), idxs

lookback = 4
X_seq, y_seq, idxs = create_lstm_input(X_scaled, y_scaled, lookback)
y_seq = pd.Series(y_seq, index=idxs)

# -----------------------------
# 5️⃣ 롤링 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2017Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 6️⃣ 모델 생성 함수
# -----------------------------
def build_lstm(input_shape):
    model = Sequential()
    model.add(LSTM(64, activation='tanh', input_shape=input_shape))
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mae')
    return model

# -----------------------------
# 7️⃣ 롤링 예측 + pred 저장
# -----------------------------
train_mae_list, test_mae_list = [], []
rolling_preds = []  # ⭐ 예측값 저장 리스트

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    train_start = train_end - (train_window - 1)
    test_end = test_start

    # LSTM용 슬라이싱
    idx_arr = np.array(idxs)

    train_mask = (idx_arr >= train_start) & (idx_arr <= train_end)
    test_mask  = (idx_arr == test_end)

    X_train, y_train = X_seq[train_mask], y_seq[train_mask]
    X_test,  y_test  = X_seq[test_mask], y_seq[test_mask]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = build_lstm((X_train.shape[1], X_train.shape[2]))
        early = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)

        model.fit(X_train, y_train, epochs=200, batch_size=8,
                  verbose=0, callbacks=[early])

        # ---- Train 예측 ----
        train_pred = model.predict(X_train, verbose=0)
        test_pred  = model.predict(X_test, verbose=0)

        # ---- 스케일 복원 ----
        train_pred_inv = scaler_y.inverse_transform(train_pred)
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1,1))

        test_pred_inv = scaler_y.inverse_transform(test_pred)
        y_test_inv  = scaler_y.inverse_transform(y_test.values.reshape(-1,1))

        # ---- MAE ----
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(test_pred_inv[0]),
            'actual': float(y_test_inv[0])
        })

    except Exception as e:
        print(f"❌ Error: {e}")
        continue

# -----------------------------
# 8️⃣ 결과 요약 + pred_df 출력
# -----------------------------
print("\n📊 LSTM 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE:  {np.mean(test_mae_list):.4f}")

# ⭐ 예측값 시계열 출력
pred_df = pd.DataFrame(rolling_preds).set_index('date')
print("\n📌 LSTM 예측값(pred) 시계열:")
print(pred_df)



📊 LSTM 롤링 예측 결과
📉 평균 Train MAE: 0.8140
📈 평균 Test MAE:  2.8509

📌 LSTM 예측값(pred) 시계열:
            pred    actual
date                      
2023Q1 -0.107376 -0.167099
2023Q2 -0.120912  1.198797
2023Q3  0.688824  4.477996
2023Q4 -1.464730 -4.657232
2024Q1 -0.835468 -2.547821
2024Q2  0.192028 -0.735633
2024Q3 -4.149986  2.741664
2024Q4  2.011340 -4.554742
2025Q1 -1.879964 -1.866263
2025Q2 -1.323711  2.712833


In [94]:
level_2025Q1 = 8297.6

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 8187.763792507172


# 단변량 아리마(예측도 포함)

In [96]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기장비lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상
# -----------------------------
y = df['전기장비 제조업'].dropna()

# -----------------------------
# 3️⃣ 롤링 윈도우
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2021Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ (p,d,q) 범위
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                y_train = y.loc[train_start:train_end]
                y_test = y.loc[test_end:test_end]

                if len(y_train) < train_window:
                    continue

                try:
                    model = ARIMA(y_train, order=(p, d, q))
                    fit = model.fit()

                    pred = fit.forecast(steps=1)

                    train_mae = mean_absolute_error(y_train, fit.fittedvalues)
                    test_mae = mean_absolute_error(y_test, pred)

                    train_mae_list.append(train_mae)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 최적 모델 선택
# -----------------------------
results_df = pd.DataFrame(results)

if results_df.empty:
    print("⚠️ 모든 조합 실패")
else:
    best = results_df.loc[results_df['test_MAE'].idxmin()]
    best_p, best_d, best_q = int(best['p']), int(best['d']), int(best['q'])

    print("✅ 최적 ARIMA(p,d,q):", (best_p, best_d, best_q))
    print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
    print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

# -----------------------------
# 8️⃣ ⭐ 최적 모델로 pred 저장 (재예측)
# -----------------------------
rolling_preds = []
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]

    if len(y_train) < train_window:
        continue

    try:
        model = ARIMA(y_train, order=(best_p, best_d, best_q))
        fit = model.fit()

        pred = fit.forecast(steps=1)

        # MAE
        train_mae_list.append(mean_absolute_error(y_train, fit.fittedvalues))
        test_mae_list.append(mean_absolute_error(y_test, pred))

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(pred),
            'actual': float(y_test.values[0])
        })

    except Exception as e:
        print("❌ 재예측 오류:", e)
        continue

# -----------------------------
# 9️⃣ pred_df 출력
# -----------------------------
pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 ARIMA 최적모델 예측(pred) 시계열:")
print(pred_df)

print("\n📉 최종 Train MAE:", np.mean(train_mae_list))
print("📈 최종 Test MAE:", np.mean(test_mae_list))


✅ 최적 ARIMA(p,d,q): (0, 1, 2)
📉 평균 Train MAE: 3.3782
📈 평균 Test MAE: 2.9442

📌 ARIMA 최적모델 예측(pred) 시계열:
            pred    actual
date                      
2022Q1  0.954324  5.842324
2022Q2  0.541721  0.267532
2022Q3  1.404535 -3.314667
2022Q4  1.261751 -3.390126
2023Q1  0.546317 -0.167099
2023Q2  0.670947  1.198797
2023Q3  0.844867  4.477996
2023Q4  1.421961 -4.657232
2024Q1 -0.693421 -2.547821
2024Q2  0.443201 -0.735633
2024Q3 -0.428712  2.741664
2024Q4  2.132440 -4.554742
2025Q1 -2.349218 -1.866263
2025Q2  0.354959  2.712833

📉 최종 Train MAE: 3.378175475725655
📈 최종 Test MAE: 2.9441769734160004


In [97]:
level_2025Q1 = 8297.6

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 8327.053062990253
